# 2.000 Train Family ile E5-LoRA + İnsan Kontrollü 600-Family Değerlendirmesi

Bu notebook Git shard havuzundaki 2.000 final train family'sini kullanır. İleri yön bütün family'lerde, ters yön ise deterministik seçilen yarısında kullanılır.

**Protokol:**
1. Repodaki tam 2.000 `accepted`/`human_review` resmî train shard family'si checksum ile sabitlenir; train içinden validation ayrılmaz.
2. Her family ileri yönde, family'lerin yarısı ayrıca ters yönde eğitilir; her örnekte iki morfolojik hard ve bir semantik hard kullanılır.
3. İnsan-kontrollü V1'den sabit, feature-dengeli 100 family yalnız development ve MRR@10 checkpoint seçimi içindir.
4. Kalan insan-kontrollü V1 500 family final değerlendirmede bir kez açılır.
5. İnsan kontrolü tamamlanmamış V2 uzantısı bu deneye alınmaz; paper sonucu en az üç seed ile tekrarlanır.


## 1. Colab kurulumu


In [ ]:
# Colab'ın önceden kurulu torchvision paketi bazı torch sürümleriyle NMS çakışması yapıyor.
# Bu text-only eğitimde torchvision ve eski torchao gerekmediği için kaldırıyoruz.
!pip uninstall -y -q torchvision torchao
!pip install -q -U "sentence-transformers>=3.1.0" "transformers>=4.48.0" peft accelerate datasets pandas

import hashlib, json, os, random, shutil, subprocess, sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch

print("torch", torch.__version__)
print("GPU", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "YOK")
assert torch.cuda.is_available(), "GPU runtime seçin."


## 2. Repo ve deney ayarları


In [ ]:
REPO_URL = "https://github.com/TR-morph-retrieval/turkish-morph-retrieval.git"
ROOT = Path("/content/turkish-morph-retrieval")
if not (ROOT / ".git").exists():
    if ROOT.exists(): shutil.rmtree(ROOT)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))

INSTRUCT = "Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:"
MODELS = {
    "modernbert-tr": {"repo":"ytu-ce-cosmos/modernbert-tr-embed", "q":INSTRUCT, "d":"",
                      "targets":["Wqkv","Wo","Wi"]},
    "bge-m3": {"repo":"BAAI/bge-m3", "q":"", "d":"",
               "targets":["query","key","value","dense"]},
    "e5-large": {"repo":"intfloat/multilingual-e5-large", "q":"query: ", "d":"passage: ",
                 "targets":["query","key","value","dense"]},
    "trmteb-ft": {"repo":"trmteb/turkish-embedding-model-fine-tuned", "q":"", "d":"",
                  "targets":["query","key","value","dense"]},
}
SELECTED_KEY = "e5-large"
CFG = MODELS[SELECTED_KEY]
SEED = 42
TRAIN_SHARD_DIR = ROOT / "train/data/shards"
TRAIN_FAMILY_LIMIT = 2000  # Tamamlanmadan eğitim başlamaz: 2.000 accepted/human_review family
TRAIN_SLOT_RANGES = None
EVAL_FILE = ROOT / "test/data/final_test/morph_test_600_human_reviewed.json"
PILOT_VALIDATION_SIZE = 100  # İnsan-kontrollü 600 aile içinden sabit, feature-dengeli development slice
RUN_SEALED_TEST = True  # İnsan-kontrollü dosyanın kalan 500 family'sinde tek final koşu
PARAMS = dict(epochs=5, early_stop_patience=2, early_stop_min_delta=.001, batch_size=32, cache_mini_batch=16, learning_rate=1e-5,
              warmup_ratio=.1, weight_decay=.01, lora_r=32, lora_alpha=64,
              lora_dropout=.05, max_seq_length=256,
              output_dir=f"/content/{SELECTED_KEY}_alltrain_seed{SEED}")
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(SELECTED_KEY, CFG["repo"], PARAMS)


## 3. Final train verisini doğrula ve development ayır


In [ ]:
from datasets import Dataset

def stable_dev(family_id, seed=SEED):
    h = hashlib.sha256(f"{seed}:{family_id}".encode()).digest()
    return int.from_bytes(h[:4], "big") % 10 == 0

def load_rows(shard_dir, limit=None, ranges=None):
    paths=sorted(shard_dir.glob("*.jsonl"))
    if not paths: raise FileNotFoundError(f"Train shard bulunamadı: {shard_dir}")
    raw=[json.loads(x) for path in paths for x in path.read_text(encoding="utf-8").splitlines() if x.strip()]
    if not raw: raise ValueError("Train JSONL boş")
    slot_id=lambda r: r.get("slot_id") or r.get("family",{}).get("slot_id")
    ids=[slot_id(r) for r in raw]
    if None in ids or len(ids)!=len(set(ids)): raise ValueError("Eksik/tekrarlı slot_id")
    slot_no=lambda r:int(slot_id(r).rsplit("_",1)[-1])
    if ranges is not None:
        raw=[r for r in raw if any(a<=slot_no(r)<=b for a,b in ranges)]
        expected={i for a,b in ranges for i in range(a,b+1)}
        actual={slot_no(r) for r in raw}
        if actual!=expected: raise ValueError(f"Train snapshot eksik/fazla: eksik={sorted(expected-actual)[:10]} fazla={sorted(actual-expected)[:10]}")
    raw=sorted(raw,key=slot_no)
    # Bu koşuda accepted ve human_review eğitime girer; reject/pilot_only kesinlikle giremez.
    raw=[r for r in raw if r.get("status") in {"accepted","human_review"} and r.get("purpose") != "pilot_only" and r.get("eligible_for_final_train") is not False]
    # Bazı eski export satırları accepted metadata taşırken gerçek family gövdesini taşımıyor.
    # Bunları sessizce eğitime sokmak yerine kimlikleriyle raporla ve uygun havuzdan çıkar.
    malformed=[slot_id(r) for r in raw if not isinstance(r.get("family"),dict)]
    load_rows.malformed_slots=malformed
    if malformed: print(f"UYARI: family gövdesi olmayan {len(malformed)} kayıt dışlandı: {malformed}")
    raw=[r for r in raw if isinstance(r.get("family"),dict)]
    # Fazla uygun kayıt varsa en küçük slot_id'li ilk N kayıt deterministik sabitlenir.
    if limit is not None and len(raw)<limit: raise ValueError(f"En az {limit} accepted/human_review final family gerekli; bulunan {len(raw)}")
    if limit is not None: raw=raw[:limit]
    rows=[]
    for r in raw:
        f=r["family"]; by={c["slot"]:c["text"].strip() for c in f["candidates"]}
        required={"positive","morph_1","morph_2","semantic_1"}
        if set(by) != required: raise ValueError(f"Aday şeması hatalı: {slot_id(r)}")
        family_id=f.get("family_id") or slot_id(r)
        if not family_id: raise ValueError(f"Kimlik eksik: {r.get('slot_id', '<bilinmiyor>')}")
        rows.append(dict(family_id=family_id, source_status=r.get("status"), target_feature=f["target_feature"],
                         query=f["query"].strip(), positive=by["positive"],
                         morph_1=by["morph_1"], morph_2=by["morph_2"],
                         semantic_1=by["semantic_1"]))
    ids=[r["family_id"] for r in rows]
    if len(ids)!=len(set(ids)): raise ValueError("Tekrarlı family_id")
    return rows

ALL_ROWS=load_rows(TRAIN_SHARD_DIR, TRAIN_FAMILY_LIMIT, TRAIN_SLOT_RANGES)
MALFORMED_SOURCE_SLOTS=getattr(load_rows,"malformed_slots",[])
TRAIN_ROWS=ALL_ROWS
DEV_ROWS=[]  # development aşağıda sealed benchmarkın sabit 100-family bölümü
TRAIN_FAMILY_LIMIT=len(TRAIN_ROWS)
assert TRAIN_FAMILY_LIMIT>0
TRAIN_SOURCE_SHA256=hashlib.sha256(json.dumps(TRAIN_ROWS,ensure_ascii=False,sort_keys=True).encode()).hexdigest()
TRAIN_STATUS_COUNTS=dict(pd.Series([r['source_status'] for r in TRAIN_ROWS]).value_counts())
print(f"final train families={len(TRAIN_ROWS)} status={TRAIN_STATUS_COUNTS} source_sha256={TRAIN_SOURCE_SHA256}")
display(pd.Series([r['target_feature'] for r in ALL_ROWS]).value_counts().describe().to_frame('fenomen dağılımı'))

# İnsan-kontrollü birleşik 600-family dosyasını sabit 100 development + 500 final test olarak ayır.
from collections import defaultdict
if not EVAL_FILE.exists(): raise FileNotFoundError(f"İnsan-kontrollü değerlendirme dosyası bulunamadı: {EVAL_FILE}")
eval_payload=json.loads(EVAL_FILE.read_text(encoding="utf-8"))
v1_eval_items=eval_payload.get("items")
if not isinstance(v1_eval_items,list): raise ValueError(f"Beklenen items dizisi yok: {EVAL_FILE}")
assert len(v1_eval_items)==600 and eval_payload.get("total_families")==600
assert all(x.get("target_split")=="sealed_test" for x in v1_eval_items)
EVAL_SOURCE_SHA256=hashlib.sha256(EVAL_FILE.read_bytes()).hexdigest()
all_eval_ids=[x["family_id"] for x in v1_eval_items]
assert len(all_eval_ids)==600 and len(set(all_eval_ids))==600
by_feature=defaultdict(list)
for item in v1_eval_items: by_feature[item["target_feature"]].append(item)
rank_key=lambda x: hashlib.sha256(f"{SEED}:{x['family_id']}".encode()).hexdigest()
PILOT_VAL_ITEMS=[sorted(group,key=rank_key)[0] for _,group in sorted(by_feature.items())]
remaining=[x for x in v1_eval_items if x not in PILOT_VAL_ITEMS]
PILOT_VAL_ITEMS += sorted(remaining,key=rank_key)[:PILOT_VALIDATION_SIZE-len(PILOT_VAL_ITEMS)]
PILOT_VAL_IDS={x["family_id"] for x in PILOT_VAL_ITEMS}
FINAL_TEST_ITEMS=[x for x in v1_eval_items if x["family_id"] not in PILOT_VAL_IDS]
assert len(PILOT_VAL_ITEMS)==PILOT_VALIDATION_SIZE and len(FINAL_TEST_ITEMS)==500
print(f"evaluation source={EVAL_FILE}")
print(f"evaluation sha256={EVAL_SOURCE_SHA256}")
print(f"evaluation split: dev={len(PILOT_VAL_ITEMS)} (insan-kontrollü, feature-dengeli), final_test={len(FINAL_TEST_ITEMS)} (insan-kontrollü)")


## 4. Model-doğru prefix ve eğitim dataset’i


In [ ]:
def qtext(x): return CFG["q"] + x
def dtext(x): return CFG["d"] + x

# Yarı simetrik augmentation: ileri yön %100, ters yön family'lerin deterministik %50'si.
reverse_order=sorted(TRAIN_ROWS,key=lambda r:hashlib.sha256(f"{SEED}:reverse:{r['family_id']}".encode()).hexdigest())
REVERSE_IDS={r['family_id'] for r in reverse_order[:len(TRAIN_ROWS)//2]}
train_samples = []
for r in TRAIN_ROWS:
    # 1. Asıl yön: query -> positive
    train_samples.append({
        "anchor": qtext(r["query"]), "positive": dtext(r["positive"]),
        "negative_1": dtext(r["morph_1"]), "negative_2": dtext(r["morph_2"]),
        "negative_3": dtext(r["semantic_1"])
    })
    # 2. Family'lerin yarısında ters yön: positive -> query
    if r['family_id'] in REVERSE_IDS:
        train_samples.append({
            "anchor": qtext(r["positive"]), "positive": dtext(r["query"]),
            "negative_1": dtext(r["morph_1"]), "negative_2": dtext(r["morph_2"]),
            "negative_3": dtext(r["semantic_1"])
        })

train_dataset = Dataset.from_list(train_samples)
assert len(train_dataset)==TRAIN_FAMILY_LIMIT+TRAIN_FAMILY_LIMIT//2
STEPS_PER_EPOCH=max(1,(len(train_dataset)+PARAMS['batch_size']-1)//PARAMS['batch_size'])
print(f"Toplam eğitim örneği: {len(train_dataset)} = {len(TRAIN_ROWS)} ileri + {len(REVERSE_IDS)} ters; adım/epoch≈{STEPS_PER_EPOCH}")
print(train_dataset[0])


## 5. Development retrieval metriği


In [ ]:
def rank_dev(model, rows, batch_size=32):
    queries=[qtext(r["query"]) for r in rows]
    docs=[]
    for r in rows: docs += [dtext(r[k]) for k in ("positive","morph_1","morph_2","semantic_1")]
    qe=model.encode(queries,batch_size=batch_size,normalize_embeddings=True,convert_to_numpy=True)
    de=model.encode(docs,batch_size=batch_size,normalize_embeddings=True,convert_to_numpy=True).reshape(len(rows),4,-1)
    scores=np.einsum("bd,bkd->bk",qe,de)
    ranks=np.array([1+np.sum(s[1:]>s[0]) for s in scores])
    return {
        "families":len(rows), "recall@1":float(np.mean(ranks<=1)),
        "recall@3":float(np.mean(ranks<=3)), "mrr":float(np.mean(1/ranks)),
        "pairwise_morph_accuracy":float(np.mean(scores[:,0,None] > scores[:,1:3])),
        "family_consistency":float(np.mean(np.all(scores[:,0,None] > scores[:,1:],axis=1))),
        "mean_rank":float(np.mean(ranks))}


## 6. Taban modeli yükle


In [ ]:
from sentence_transformers import SentenceTransformer

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model=SentenceTransformer(CFG["repo"],trust_remote_code=True,
                          model_kwargs={"torch_dtype":dtype})
model.max_seq_length=PARAMS["max_seq_length"]
from sentence_transformers.evaluation import SentenceEvaluator
from test.evaluation import score_encoder

class ControlledRecallEvaluator(SentenceEvaluator):
    def __init__(self, items, name="pilot_validation"):
        self.items, self.name = items, name
        self.primary_metric=f"{name}_mrr_at_10"
        self.greater_is_better=True
    def __call__(self, model, output_path=None, epoch=-1, steps=-1):
        result=score_encoder(model,self.items,query_prefix=CFG["q"],document_prefix=CFG["d"],batch_size=32)
        summary, score_map = result["summary"], result["scores"]
        # 11 aday içindeki gold softmax olasılığı: train loss ile aynı değil,
        # fakat validation için sabit ve yorumlanabilir bir controlled ranking loss.
        losses=[]
        for item in self.items:
            values=np.asarray([score_map[item["family_id"]][c["id"]] for c in item["candidates"]], dtype=float) * 20.0
            gold_index=next(i for i,c in enumerate(item["candidates"]) if c["id"] == item["gold_id"])
            shifted=values-values.max()
            losses.append(float(-shifted[gold_index]+np.log(np.exp(shifted).sum())))
        return {f"{self.name}_controlled_loss":float(np.mean(losses)),
                f"{self.name}_recall_at_1":summary["recall@1"],
                f"{self.name}_recall_at_3":summary["recall@3"],
                f"{self.name}_mrr_at_10":summary["mrr@10"]}

pilot_evaluator=ControlledRecallEvaluator(PILOT_VAL_ITEMS)
BASE_VAL=score_encoder(model,PILOT_VAL_ITEMS,query_prefix=CFG["q"],document_prefix=CFG["d"],batch_size=32)["summary"]
BASE_TEST=score_encoder(model,FINAL_TEST_ITEMS,query_prefix=CFG["q"],document_prefix=CFG["d"],batch_size=32)["summary"]
display(pd.DataFrame([{"stage":"zero-shot / validation_100",**BASE_VAL},
                      {"stage":"zero-shot / final_test_500",**BASE_TEST}]))
BASE_DEV=BASE_VAL


## 7. LoRA’yı doğru modüle bağla ve cached contrastive loss kur


In [ ]:
from peft import LoraConfig, get_peft_model
from sentence_transformers.sentence_transformer.losses import CachedMultipleNegativesRankingLoss

first=model._first_module()
available={name.rsplit('.',1)[-1] for name, _ in first.auto_model.named_modules()}
targets=[name for name in CFG["targets"] if name in available]
if not targets: raise RuntimeError(f"LoRA hedefi bulunamadı. Kullanılabilir adlar: {sorted(available)[:40]}")
print("LoRA targets:", targets)
peft_model=get_peft_model(first.auto_model,LoraConfig(
    r=PARAMS["lora_r"], lora_alpha=PARAMS["lora_alpha"],
    lora_dropout=PARAMS["lora_dropout"], bias="none",
    target_modules=targets, task_type="FEATURE_EXTRACTION"))
first.auto_model=peft_model  # kritik: wrapper SentenceTransformer'a geri bağlanır
peft_model.print_trainable_parameters()
trainable=sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
total=sum(p.numel() for p in peft_model.parameters())
TRAINABLE_PERCENT=100.0*trainable/total
print(f"Eğitilebilir LoRA oranı: %{TRAINABLE_PERCENT:.2f}")
assert 1.8 <= TRAINABLE_PERCENT <= 3.5, f"Beklenen %2–3 LoRA bandı dışında: %{TRAINABLE_PERCENT:.2f}"
loss=CachedMultipleNegativesRankingLoss(model,scale=20.0,
                                         mini_batch_size=PARAMS["cache_mini_batch"])
print("Cached MNRL: üç açık hard negatif + büyük etkili batch")


## 8. Eğit


In [ ]:
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.training_args import BatchSamplers
from transformers import EarlyStoppingCallback, TrainerCallback

# PEFT ile bazı Colab sürümlerinde Trainer en iyi checkpoint'i dışarıdaki model
# nesnesine geri yüklemeyebiliyor. Sadece LoRA ağırlıklarının en iyi halini ayrıca saklarız.
class RestoreBestLoRA(TrainerCallback):
    def __init__(self, metric_name, greater_is_better=False):
        self.metric_name, self.greater_is_better = metric_name, greater_is_better
        self.best_score = float('-inf') if greater_is_better else float('inf')
        self.best_state, self.best_epoch = None, None
    def on_evaluate(self, args, state, control, metrics=None, model=None, **kwargs):
        metrics = metrics or {}
        score = metrics.get(f'eval_{self.metric_name}', metrics.get(self.metric_name))
        is_better = score is not None and (score > self.best_score if self.greater_is_better else score < self.best_score)
        if is_better:
            self.best_score, self.best_epoch = float(score), state.epoch
            self.best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if 'lora_' in k}
        return control

best_lora = RestoreBestLoRA(pilot_evaluator.primary_metric, greater_is_better=True)

args=SentenceTransformerTrainingArguments(
    output_dir=PARAMS["output_dir"], num_train_epochs=PARAMS["epochs"],
    per_device_train_batch_size=PARAMS["batch_size"],
    learning_rate=PARAMS["learning_rate"], warmup_ratio=PARAMS["warmup_ratio"],
    weight_decay=PARAMS["weight_decay"],
    fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
    batch_sampler=BatchSamplers.NO_DUPLICATES, logging_strategy="steps", logging_steps=max(1,STEPS_PER_EPOCH//4), logging_first_step=True,
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model=pilot_evaluator.primary_metric,
    greater_is_better=True, eval_on_start=True, seed=SEED, data_seed=SEED, report_to="none")
trainer=SentenceTransformerTrainer(model=model,args=args,train_dataset=train_dataset,loss=loss,
    evaluator=pilot_evaluator,callbacks=[EarlyStoppingCallback(
        early_stopping_patience=PARAMS["early_stop_patience"],
        early_stopping_threshold=PARAMS["early_stop_min_delta"] ), best_lora])
result=trainer.train()
if best_lora.best_state:
    trainer.model.load_state_dict(best_lora.best_state, strict=False)
model = trainer.model
print(f'En iyi LoRA: epoch={best_lora.best_epoch}, validation MRR@10={best_lora.best_score:.4f}')
print(result)

# Trainer'ın ham geçmişini anlaşılır iki tablo halinde yazdır.
history=pd.DataFrame(trainer.state.log_history)
train_log=history[history.get("loss", pd.Series(index=history.index, dtype=float)).notna()]
if not train_log.empty:
    display(train_log[[c for c in ["epoch", "step", "loss", "learning_rate"] if c in train_log]].rename(columns={"loss":"training_loss"}))
val_mrr_key=f"eval_{pilot_evaluator.primary_metric}"
controlled_loss_key="eval_pilot_validation_controlled_loss"
val_log=history[history.get(val_mrr_key, pd.Series(index=history.index, dtype=float)).notna()]
if not val_log.empty:
    cols=[c for c in ["epoch", controlled_loss_key, "eval_pilot_validation_recall_at_1", "eval_pilot_validation_recall_at_3", val_mrr_key] if c in val_log]
    display(val_log[cols].rename(columns={controlled_loss_key:"controlled_validation_loss", "eval_pilot_validation_recall_at_1":"validation_recall@1", "eval_pilot_validation_recall_at_3":"validation_recall@3", val_mrr_key:"validation_mrr@10"}))


## 9. Eğitim tamamlandı


In [ ]:
FT_DEV=None
print("Eğitim bitti. Sonraki hücre 100-family development ve 500-family insan-kontrollü final değerlendirmeyi çalıştıracak.")


## 10. Sabit 100-family validation + 500-family insan-kontrollü final değerlendirme


In [ ]:
PILOT_VAL_RESULT=score_encoder(model,PILOT_VAL_ITEMS,query_prefix=CFG["q"],document_prefix=CFG["d"],batch_size=32)["summary"]
FT_DEV=PILOT_VAL_RESULT  # manifest'te eğitim sonrası doğrulama sonucunu da sakla
SEALED_RESULT=score_encoder(model,FINAL_TEST_ITEMS,query_prefix=CFG["q"],document_prefix=CFG["d"],batch_size=32)["summary"] if RUN_SEALED_TEST else None
rows=[{"stage":"zero-shot / validation_100", **BASE_VAL},
      {"stage":"best LoRA / validation_100", **PILOT_VAL_RESULT}]
if SEALED_RESULT:
    rows += [{"stage":"zero-shot / final_test_500", **BASE_TEST},
             {"stage":"best LoRA / final_test_500", **SEALED_RESULT}]
display(pd.DataFrame(rows))


## 11. Model ve deney manifestini kaydet


In [ ]:
out=Path(PARAMS["output_dir"]); save_path=out/"alltrain_model"
model.save(str(save_path))
manifest={"model":CFG["repo"],"selected_key":SELECTED_KEY,"seed":SEED,
          "params":PARAMS,"train_source":str(TRAIN_SHARD_DIR),
          "train_source_sha256":TRAIN_SOURCE_SHA256,"train_slot_ranges":TRAIN_SLOT_RANGES,
          "train_families":len(TRAIN_ROWS),"train_status_counts":TRAIN_STATUS_COUNTS,"malformed_source_slots_excluded":MALFORMED_SOURCE_SLOTS,"dev_families":PILOT_VALIDATION_SIZE,
          "base_dev":BASE_DEV,"fine_tuned_dev":FT_DEV,
          "train_pairs":len(train_dataset),"reverse_train_families":len(REVERSE_IDS),"trainable_percent":TRAINABLE_PERCENT,"pilot_mode":False,"pilot_validation_size":PILOT_VALIDATION_SIZE,
          "pilot_validation_family_ids":sorted(PILOT_VAL_IDS),
          "pilot_validation_summary":PILOT_VAL_RESULT,
          "evaluation_source":str(EVAL_FILE),"evaluation_source_sha256":EVAL_SOURCE_SHA256,
          "evaluation_total_families":600,"final_test_families":len(FINAL_TEST_ITEMS),
          "evaluation_human_review_complete":True,"v2_extension_included":False,
          "sealed_test_opened":RUN_SEALED_TEST,
          "sealed_remainder_summary":SEALED_RESULT if SEALED_RESULT else None}
def json_default(value):
    if isinstance(value,np.generic): return value.item()
    if isinstance(value,np.ndarray): return value.tolist()
    if isinstance(value,Path): return str(value)
    raise TypeError(f"JSON'a çevrilemeyen tip: {type(value).__name__}")
(out/"experiment_manifest.json").write_text(json.dumps(manifest,ensure_ascii=False,indent=2,default=json_default))
print("Kaydedildi:",save_path)


## 12. Haricî genel retrieval koruma testi (TabiBench Information Retrieval)

Bu bölüm checkpoint seçimine **katılmaz** ve MorphTR sealed sonuçlarına geri beslenmez. Resmî TabiBERT kodunun kullandığı beş Türkçe retrieval veri kaynağında base E5 ile MorphTR-LoRA E5'i aynı query–corpus nDCG@10 protokolüyle karşılaştırır. Bu bir TabiBench leaderboard yeniden-eğitimi değil, genel retrieval yeteneğinin korunup korunmadığını ölçen zero-shot dış testtir. `BiText`, belge retrieval'dan farklı bir bitext-mining görevi olduğu için bu nDCG tablosuna karıştırılmaz.


In [ ]:
RUN_TABIBENCH_IR = True
TABIBENCH_IR_DATASETS = {
    "MSMarco-TR":"trmteb/msmarco-tr_fine_tuning_dataset",
    "SciFact-TR":"trmteb/scifact-tr_fine_tuning_dataset",
    "FiQA-TR":"trmteb/fiqa-tr_fine_tuning_dataset",
    "NFCorpus-TR":"trmteb/nfcorpus-tr_fine_tuning_dataset",
    "Quora-TR":"trmteb/quora-tr_fine_tuning_dataset",
}

if RUN_TABIBENCH_IR:
    import gc
    from collections import defaultdict
    from datasets import load_dataset

    def retrieval_view(split):
        anchor2pos=defaultdict(set); skipped=0
        for ex in split:
            anchor=ex.get("anchor"); positive=ex.get("positive")
            if not isinstance(anchor,str) or not isinstance(positive,str):
                skipped+=1; continue
            anchor=anchor.strip(); positive=positive.strip()
            if not anchor or not positive:
                skipped+=1; continue
            anchor2pos[anchor].add(positive)
        if not anchor2pos: raise ValueError("Geçerli anchor-positive çifti bulunamadı")
        if skipped: print(f"Boş/geçersiz {skipped} retrieval çifti atlandı.")
        queries=list(anchor2pos)
        corpus=list(dict.fromkeys(doc for q in queries for doc in anchor2pos[q]))
        did={doc:i for i,doc in enumerate(corpus)}
        qrels=[{did[doc] for doc in anchor2pos[q]} for q in queries]
        return queries,corpus,qrels

    def ndcg10_from_embeddings(Q,D,qrels,query_chunk=256):
        values=[]; k=min(10,len(D))
        for start in range(0,len(Q),query_chunk):
            scores=Q[start:start+query_chunk] @ D.T
            top=np.argpartition(-scores,k-1,axis=1)[:,:k]
            for local,row_idx in enumerate(top):
                row_scores=scores[local,row_idx]; ranked=row_idx[np.argsort(-row_scores)]
                rel=qrels[start+local]
                dcg=sum(1.0/np.log2(rank+2.0) for rank,di in enumerate(ranked) if int(di) in rel)
                ideal=min(len(rel),k); idcg=sum(1.0/np.log2(rank+2.0) for rank in range(ideal))
                values.append(dcg/idcg if idcg else 0.0)
        return float(np.mean(values))

    def external_ir_score(st_model,dataset_path):
        ds=load_dataset(dataset_path)
        split=ds["test"] if "test" in ds else ds["validation"]
        queries,corpus,qrels=retrieval_view(split)
        q_prefix=CFG.get("q") or ""; d_prefix=CFG.get("d") or ""
        Q=st_model.encode([q_prefix+x for x in queries],batch_size=32,normalize_embeddings=True,convert_to_numpy=True,show_progress_bar=True)
        D=st_model.encode([d_prefix+x for x in corpus],batch_size=32,normalize_embeddings=True,convert_to_numpy=True,show_progress_bar=True)
        return ndcg10_from_embeddings(Q,D,qrels),len(queries),len(corpus)

    lora_rows=[]
    for task,path in TABIBENCH_IR_DATASETS.items():
        score,nq,nd=external_ir_score(model,path)
        lora_rows.append({"task":task,"model_stage":"MorphTR-LoRA","ndcg@10":score,"queries":nq,"corpus":nd})
    del trainer, loss, peft_model, first, model
    gc.collect(); torch.cuda.empty_cache()
    base_ir_model=SentenceTransformer(CFG["repo"],trust_remote_code=True,model_kwargs={"torch_dtype":dtype})
    base_ir_model.max_seq_length=PARAMS["max_seq_length"]
    base_rows=[]
    for task,path in TABIBENCH_IR_DATASETS.items():
        score,nq,nd=external_ir_score(base_ir_model,path)
        base_rows.append({"task":task,"model_stage":"Base E5","ndcg@10":score,"queries":nq,"corpus":nd})
    ir_table=pd.DataFrame(base_rows+lora_rows)
    wide=ir_table.pivot(index="task",columns="model_stage",values="ndcg@10")
    wide["delta"]=wide["MorphTR-LoRA"]-wide["Base E5"]
    display(wide.reset_index())
    ir_table.to_csv(out/"tabibench_ir_base_vs_lora.csv",index=False)
